# Spatial Cost Inputs

Generates a per-location, per-technology cost overrides CSV for the global run.

The output CSV has columns: `lat, lon, tech, interest_rate, build_cost_multiplier,
land_cost_usd_per_km2_year, water_cost_usd_per_m3`.

**Build cost multiplier logic (depth-based):**
- **Onshore / coastal cells** (`onshore_land_pct > 0`): `build_cost_multiplier = 1.0`.
- **Offshore cells** (`onshore_land_pct == 0`): multiplier varies by bathymetry depth
  using a piecewise-linear function, with separate curves for **wind** and **plant equipment**:
  - Shallow / fixed-bottom (0–60 m): wind 1.3–1.8×, plant 1.1–1.3×
  - Floating (60–300 m): wind 1.8–2.5×, plant 1.3–1.8×
  - Deep floating / early unmoored (300–1500 m): wind 2.5–3.5×, plant 1.8–2.5×
  - Ultra-deep / unmoored vessel-based (>1500 m): wind 3.5×, plant 2.5×

**Composability:** The depth multiplier is one factor in a planned composite:
`build_cost_multiplier = depth_mult × remoteness_mult × labour_cost_mult`.
Remoteness and labour cost factors will be added in a future step.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'model').is_dir():
            return p
    raise RuntimeError("Could not locate repo root containing 'model/'")


NOTEBOOK_DIR = Path('__file__').resolve().parent if '__file__' in globals() else Path.cwd()
repo_root = find_repo_root(NOTEBOOK_DIR)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', repo_root)

In [ ]:
# ----------------
# USER PARAMETERS
# ----------------

TECH_YAML = repo_root / 'inputs' / 'tech_config_ammonia_plant_2030_dea.yaml'
MAX_CAPACITIES_CSV = repo_root / 'data' / '20251222_max_capacities.csv'

# Output path (this file is consumed by the global run)
OUTPUT_CSV = repo_root / 'inputs' / 'spatial_cost_inputs.csv'

# Default water cost (applied uniformly; override per-location if needed)
DEFAULT_WATER_COST_USD_PER_M3 = 2.0

# Default land cost (only meaningful for cells with land)
DEFAULT_LAND_COST_USD_PER_KM2_YEAR = 0.0

# -------------------------------------------------------
# DEPTH-BASED BUILD COST MULTIPLIER BREAKPOINTS
# -------------------------------------------------------
# Piecewise-linear interpolation: np.interp(depth, DEPTH_BP, MULT_BP)
# Breakpoints are (depth_m, multiplier) anchors.  Between anchors, values
# are linearly interpolated; beyond the last anchor, the value is clamped.
#
# Sources / rationale:
#   - Fixed-bottom max depth ~60 m (DNV, IEA Wind Task 26)
#   - Semi-sub / spar floating demonstrated to ~300 m, design envelope to ~1000 m
#   - Beyond ~1500 m: speculative unmoored / vessel-based installation
#   - Archived DEA proxy: fixed offshore ≈ 2.03× onshore, floating ≈ 2.38× onshore
#
# Wind generators carry the largest offshore premium (foundation, installation
# vessels, subsea cabling).  Plant equipment (electrolysers, batteries, H2 storage,
# ammonia synthesis) has a smaller premium driven by marine logistics.

DEPTH_BREAKPOINTS_M = [0, 60, 300, 1500]

# Wind: steep curve — foundation type changes with depth
WIND_MULTIPLIERS = [1.3, 1.8, 2.5, 3.5]

# Plant equipment: gentler curve — logistics / marinisation premium
PLANT_MULTIPLIERS = [1.1, 1.3, 1.8, 2.5]

# Techs that use the wind multiplier curve; everything else uses plant curve
WIND_TECHS = {'wind'}

print(f'Tech YAML: {TECH_YAML}')
print(f'Max capacities: {MAX_CAPACITIES_CSV}')
print(f'Output CSV: {OUTPUT_CSV}')
print(f'Depth breakpoints (m): {DEPTH_BREAKPOINTS_M}')
print(f'Wind multipliers:  {WIND_MULTIPLIERS}')
print(f'Plant multipliers: {PLANT_MULTIPLIERS}')

In [ ]:
# Load YAML tech config to get tech names and default interest rates
cfg = yaml.safe_load(TECH_YAML.read_text())
techs_cfg = cfg.get('techs', {})

# Build a list of (tech_name, interest_rate) from the YAML
tech_rates = {}
for name, params in techs_cfg.items():
    if isinstance(params, dict) and 'interest_rate' in params:
        tech_rates[name] = float(params['interest_rate'])

print(f'Found {len(tech_rates)} techs with interest rates:')
for t, r in sorted(tech_rates.items()):
    print(f'  {t}: {r:.4f}')

In [ ]:
# Load max capacities to get all locations and their land fraction
mc = pd.read_csv(MAX_CAPACITIES_CSV)

# Filter to cells with capacity
if 'max_capacity_mw' in mc.columns:
    mc = mc[mc['max_capacity_mw'] > 0].copy()

print(f'Total grid cells: {len(mc):,}')
print(f'  Pure ocean (land_pct == 0): {(mc.onshore_land_pct == 0).sum():,}')
print(f'  Coastal (0 < land_pct < 100): {((mc.onshore_land_pct > 0) & (mc.onshore_land_pct < 100)).sum():,}')
print(f'  Pure land (land_pct == 100): {(mc.onshore_land_pct == 100).sum():,}')

In [ ]:
# -------------------------------------------------------
# DEPTH MULTIPLIER FUNCTIONS
# -------------------------------------------------------
# Composable: depth_mult is one factor in a future composite
#   build_cost_multiplier = depth_mult * remoteness_mult * labour_cost_mult


def depth_build_multiplier(depth_m: float, tech: str) -> float:
    """Piecewise-linear depth -> build cost multiplier.

    Returns 1.0 for onshore (depth <= 0 or NaN).
    For offshore, interpolates between breakpoints with separate
    curves for wind vs plant equipment.
    """
    if np.isnan(depth_m) or depth_m <= 0:
        return 1.0
    if tech in WIND_TECHS:
        return float(np.interp(depth_m, DEPTH_BREAKPOINTS_M, WIND_MULTIPLIERS))
    return float(np.interp(depth_m, DEPTH_BREAKPOINTS_M, PLANT_MULTIPLIERS))


def depth_build_multiplier_vectorised(
    depths: np.ndarray, tech: str
) -> np.ndarray:
    """Vectorised version for fast DataFrame application."""
    mults = np.ones_like(depths, dtype=float)
    offshore = ~np.isnan(depths) & (depths > 0)
    if tech in WIND_TECHS:
        mults[offshore] = np.interp(depths[offshore], DEPTH_BREAKPOINTS_M, WIND_MULTIPLIERS)
    else:
        mults[offshore] = np.interp(depths[offshore], DEPTH_BREAKPOINTS_M, PLANT_MULTIPLIERS)
    return mults


# Quick sanity check
test_depths = [0, 30, 60, 150, 300, 800, 1500, 4000]
print('Depth (m)  | Wind mult | Plant mult')
print('-' * 40)
for d in test_depths:
    w = depth_build_multiplier(d, 'wind')
    p = depth_build_multiplier(d, 'electrolysis')
    print(f'{d:>9}  | {w:>9.2f} | {p:>10.2f}')

In [ ]:
# Build the spatial cost inputs DataFrame.
# One row per (location, tech) combination.
# Offshore cells get depth-dependent build cost multipliers;
# onshore / coastal cells get 1.0.

tech_names = sorted(tech_rates.keys())

# Pre-extract arrays for vectorised multiplier computation
lats = mc['latitude'].to_numpy()
lons = mc['longitude'].to_numpy()
land_pcts = mc['onshore_land_pct'].to_numpy()

# Use absolute depth for offshore cells; bathymetry_depth_m is positive = deep
if 'bathymetry_depth_m' in mc.columns:
    raw_depths = mc['bathymetry_depth_m'].to_numpy().astype(float)
else:
    raw_depths = np.full(len(mc), np.nan)

# For offshore cells (land_pct == 0), use the depth value; for land/coastal, force 0
# so the multiplier function returns 1.0.
effective_depths = np.where(land_pcts == 0, np.abs(raw_depths), 0.0)

# Build per-tech multiplier arrays and assemble rows
all_rows = []
for tech in tech_names:
    mults = depth_build_multiplier_vectorised(effective_depths, tech)
    rate = tech_rates[tech]
    for i in range(len(mc)):
        all_rows.append({
            'lat': lats[i],
            'lon': lons[i],
            'tech': tech,
            'interest_rate': rate,
            'build_cost_multiplier': round(mults[i], 4),
            'land_cost_usd_per_km2_year': DEFAULT_LAND_COST_USD_PER_KM2_YEAR,
            'water_cost_usd_per_m3': DEFAULT_WATER_COST_USD_PER_M3,
        })

sci_df = pd.DataFrame(all_rows)
print(f'Generated {len(sci_df):,} rows ({len(mc):,} locations x {len(tech_names)} techs)')

# Summarise offshore vs onshore
offshore_rows = sci_df[sci_df['build_cost_multiplier'] > 1.0]
print(f'Offshore rows (multiplier > 1): {len(offshore_rows):,} ({len(offshore_rows) // len(tech_names):,} locations)')
print(f'Onshore rows (multiplier == 1): {len(sci_df) - len(offshore_rows):,}')

# Show multiplier distribution for wind vs a plant tech
wind_offshore = sci_df[(sci_df['tech'] == 'wind') & (sci_df['build_cost_multiplier'] > 1)]
elec_offshore = sci_df[(sci_df['tech'] == 'electrolysis') & (sci_df['build_cost_multiplier'] > 1)]
print(f'\nWind offshore multiplier stats:')
print(wind_offshore['build_cost_multiplier'].describe().to_string())
print(f'\nElectrolysis offshore multiplier stats:')
print(elec_offshore['build_cost_multiplier'].describe().to_string())

In [ ]:
# Preview offshore entries by depth band (wind vs plant)
wind_rows = sci_df[sci_df['tech'] == 'wind'].copy()
elec_rows = sci_df[sci_df['tech'] == 'electrolysis'].copy()

# Merge depth info back for display
depth_lookup = mc[['latitude', 'longitude', 'bathymetry_depth_m']].copy()
depth_lookup.columns = ['lat', 'lon', 'depth_m']
wind_rows = wind_rows.merge(depth_lookup, on=['lat', 'lon'], how='left')
elec_rows = elec_rows.merge(depth_lookup, on=['lat', 'lon'], how='left')

print('Sample WIND entries by depth:')
wind_offshore = wind_rows[wind_rows['build_cost_multiplier'] > 1].sort_values('depth_m')
display(wind_offshore.head(5))
display(wind_offshore.tail(5))

print('\nSample ELECTROLYSIS entries by depth:')
elec_offshore = elec_rows[elec_rows['build_cost_multiplier'] > 1].sort_values('depth_m')
display(elec_offshore.head(5))
display(elec_offshore.tail(5))

print('\nSample onshore entries:')
display(sci_df[sci_df['build_cost_multiplier'] == 1.0].head(10))

In [ ]:
# Write to CSV
sci_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved spatial cost inputs to: {OUTPUT_CSV}')
print(f'File size: {OUTPUT_CSV.stat().st_size / 1024 / 1024:.1f} MB')